# Souty gesture recognizer retraining

Retrains MediaPipe's gesture recognizer using the HaGRID dataset instead of your own photos.

**Setup before running:**
1. Settings -> Accelerator -> GPU.
2. Add Data -> attach `innominate817/hagrid-classification-512p-no-gesture` (it already bundles the gesture classes and the `no_gesture` folder together, so the plain `hagrid-classification-512p` dataset is not needed).
3. Run all cells top to bottom.
4. When done, download `exported_model/gesture_recognizer.task` from the Output panel (top right).
5. Locally: copy that file to `../public/models/gesture_recognizer.task`.

**Why Python 3.10, not the notebook's own kernel:** `mediapipe-model-maker` depends on `tf-models-official==2.11.6`, which pins `pyyaml<6.0` - a version with no prebuilt wheel for Python 3.12 (Kaggle's current default). Building it from source currently fails there - a live, unresolved upstream bug ([mediapipe-samples#643](https://github.com/google-ai-edge/mediapipe-samples/issues/643)). Installing Python 3.10 alongside the default and running under that is the confirmed community workaround, so training runs as a subprocess under python3.10 rather than in the notebook's own cells.

**IMPORTANT:** HaGRID has no gesture matching `ILoveYou` (thumb+index+pinky). Retraining replaces the whole class set, so `ILoveYou` will NOT exist in the resulting model unless you also upload ~100 of your own photos of that hand shape as a separate Kaggle Dataset and add them in `kaggle_pipeline.py`'s `build_dataset()`. If you skip that, remove the `ILoveYou` entry from `lib/gestureDictionary.ts` after deploying this model.

HaGRID is CC BY-SA 4.0 (variant) licensed — credit it if you mention your training data anywhere public (https://github.com/hukenovs/hagrid).

## 1. Install Python 3.10 and mediapipe-model-maker under it

In [ ]:
!sudo apt-get install -y python3.10 python3.10-distutils python3.10-venv
!wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py
!python3.10 /tmp/get-pip.py
!python3.10 -m pip install mediapipe-model-maker

## 2. Force a GPU-enabled TensorFlow install

`mediapipe-model-maker` pulls in plain `tensorflow`, which needs CUDA/cuDNN already present and version-matched on the system - unreliable for a manually-installed python3.10 that isn't Kaggle's own preconfigured environment. `tensorflow[and-cuda]` bundles the matching NVIDIA libraries as pip packages instead, so it works regardless of what's on the host. Pinned to `2.15.0.post1` specifically: plain `2.15.1` (what step 1 installed) has a known broken `tensorrt` dependency in the `[and-cuda]` extra that `.post1` fixed.

In [ ]:
!python3.10 -m pip install "tensorflow[and-cuda]==2.15.0.post1"

## 3. Verify GPU is actually visible under python3.10

If this still prints an empty list, training will fall back to CPU (much slower, but not broken) - check Settings -> Accelerator is set to GPU and re-run from step 1 if you changed it after starting.

In [ ]:
!python3.10 -c "import tensorflow as tf; print('GPUs:', tf.config.list_physical_devices('GPU'))"

## 4. Write the training script

Same content as `kaggle_pipeline.py` in this repo - keep them in sync if you edit one. `HAGRID_ROOT` is confirmed against this dataset's actual mount path (`/kaggle/input/datasets/innominate817/hagrid-classification-512p-no-gesture`, found via `find /kaggle/input -maxdepth 3`); `find_class_dir()` searches one level deeper too since Kaggle sometimes nests an extra duplicate-name folder, so it doesn't matter which of those two layouts this dataset actually uses.

In [ ]:
%%writefile /kaggle/working/_train.py
"""Retrains the gesture recognizer using the HaGRID dataset. Run under
python3.10 - see the notebook markdown cells above for why."""

import shutil
from pathlib import Path

from mediapipe_model_maker import gesture_recognizer

HAGRID_ROOT = Path("/kaggle/input/datasets/innominate817/hagrid-classification-512p-no-gesture")
OUT_DIR = Path("/kaggle/working/dataset")

# Cap per class: HaGRID has 10k+ images per class, far more than needed and
# would make training slow on Kaggle's session limits.
MAX_PER_CLASS = 500

# HaGRID source folder name -> our GESTURE_WORD_MAP label
# (must match lib/gestureDictionary.ts exactly).
LABEL_MAP = {
    "palm": "Open_Palm",
    "fist": "Closed_Fist",
    "like": "Thumb_Up",
    "dislike": "Thumb_Down",
    "peace": "Victory",
    "one": "Pointing_Up",
    "ok": "OK_Sign",
    # verify "three" is the 3-finger shape you want; HaGRID also has
    # "three2" as an alternate 3-finger pose - peek at sample images first.
    "three": "Three_Fingers",
    "four": "Four_Fingers",
    "call": "Shaka_Sign",
    # No HaGRID equivalent for ILoveYou - see the note at the top.
}


def find_class_dir(root: Path, name: str) -> Path | None:
    """Looks for a folder named `name` directly under root, or one level
    deeper - covers both a flat mount and Kaggle's occasional extra
    duplicate-name nesting, without walking the whole (500k+ file) tree."""
    direct = root / name
    if direct.is_dir():
        return direct
    for sub in root.iterdir():
        if sub.is_dir():
            candidate = sub / name
            if candidate.is_dir():
                return candidate
    return None


def copy_capped(src: Path, dst: Path, cap: int) -> int:
    dst.mkdir(parents=True, exist_ok=True)
    files = sorted(src.glob("*"))[:cap]
    for f in files:
        shutil.copy(f, dst / f.name)
    return len(files)


def build_dataset() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for src_name, label in LABEL_MAP.items():
        src = find_class_dir(HAGRID_ROOT, src_name)
        if src is None:
            print(f"WARNING: no '{src_name}' folder found under {HAGRID_ROOT} - skipping {label}")
            continue
        count = copy_capped(src, OUT_DIR / label, MAX_PER_CLASS)
        print(f"{label}: {count} images (from {src})")

    none_src = find_class_dir(HAGRID_ROOT, "no_gesture")
    if none_src is None:
        raise SystemExit(f"no 'no_gesture' folder found under {HAGRID_ROOT} - cannot train without a none class")
    none_count = copy_capped(none_src, OUT_DIR / "none", MAX_PER_CLASS)
    print(f"none: {none_count} images (from {none_src})")

    # Optional: uncomment if you uploaded your own ILoveYou photos as a
    # separate Kaggle Dataset.
    # copy_capped(Path("/kaggle/input/your-ily-photos"), OUT_DIR / "ILoveYou", MAX_PER_CLASS)


def train_and_export() -> None:
    data = gesture_recognizer.Dataset.from_folder(
        dirname=str(OUT_DIR),
        hparams=gesture_recognizer.HandDataPreprocessingParams(),
    )
    train_data, rest_data = data.split(0.8)
    validation_data, test_data = rest_data.split(0.5)

    hparams = gesture_recognizer.HParams(export_dir="/kaggle/working/exported_model", epochs=20)
    options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
    model = gesture_recognizer.GestureRecognizer.create(
        train_data=train_data,
        validation_data=validation_data,
        options=options,
    )

    loss, accuracy = model.evaluate(test_data, batch_size=1)
    print(f"Test loss: {loss:.4f}, test accuracy: {accuracy:.4f}")
    # Check per-class results here too - if two classes (e.g. Shaka_Sign vs
    # ILoveYou, or Four_Fingers vs Open_Palm) are getting confused for each
    # other, that pair is too visually similar - dropping one is more
    # reliable than trying to train through it.

    model.export_model()
    print("Download /kaggle/working/exported_model/gesture_recognizer.task from the Output panel.")


if __name__ == "__main__":
    build_dataset()
    train_and_export()

## 5. Run it

`MPLBACKEND=Agg` overrides the notebook kernel's own `MPLBACKEND` value (set for its inline plotting), which would otherwise leak into this subprocess and crash matplotlib - it's only pulled in transitively by mediapipe's drawing utilities here, not actually used for output, so a plain headless backend is fine.

A `cannot import name 'runtime_version' from 'google.protobuf'` error may print partway through - that's from an optional `tensorflow_datasets` submodule failing to import; it's caught internally and does not stop the script, so it's safe to ignore.

In [ ]:
!MPLBACKEND=Agg python3.10 /kaggle/working/_train.py